In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

In [3]:
load_dotenv()
api_key=os.getenv("OPENAI_API_KEY")
client=OpenAI(api_key=api_key)

In [5]:
vector_store = client.vector_stores.create(name="events_taxonomy_space")
taxonomy_vector_id=vector_store.id
print(taxonomy_vector_id)

vs_693087a5f7b88191aea544ef5ab0560b


In [4]:
client.vector_stores.delete(vector_store_id="vs_69307ff75b44819186ecbba841506630")

VectorStoreDeleted(id='vs_69307ff75b44819186ecbba841506630', deleted=True, object='vector_store.deleted')

In [6]:
vector_store_id="vs_693087a5f7b88191aea544ef5ab0560b"

client.vector_stores.files.upload_and_poll(
    vector_store_id=vector_store_id,
    file=open("eventss.json", "rb")
)

VectorStoreFile(id='file-7KbHko5oWe6MNftRw11WVf', created_at=1764788167, last_error=None, object='vector_store.file', status='completed', usage_bytes=4346, vector_store_id='vs_693087a5f7b88191aea544ef5ab0560b', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))

In [4]:
SYSTEM_PROMPT="""You are an industry-aware event extractor.
Your job: Read the article carefully and identify **ONLY the most relevant catalytic events** described for EACH shortlisted industry provided by the user.

Definition — “Most relevant event”:
The event classification from eventss.json that best represents the **primary catalytic action** affecting that specific industry/sub-category in the article. 
If multiple events could apply, select only the ones with the strongest evidence, largest impact, or clearest link to the industry.
- Always make deterministic choices: given identical evidence, the same industry must always get the same event type.
- Avoid switching between plausible taxonomy entries across different runs.


Rules & Best Practices:

1) Taxonomy Grounding:
- You MUST rely exclusively on eventss.json for output. Each event type in eventss.json includes a set of keywords. These keywords act as semantic anchors.  
- When matching article text to event types, you MUST:
- Recognize synonyms, paraphrases, or phrases with equivalent meaning to the listed keywords.
- Avoid literal keyword dependency,use semantic equivalence, not exact string matches.
- Prioritize the event type whose description + keyword family is the strongest semantic match.
- Use the **exact event_type strings from the eventss.json taxonomy**.
- Do NOT invent new event types or modify existing ones.
- Only assign events explicitly tied to the shortlisted industry/sub-category.
- If no event is relevant for an industry → output an empty events array and event_frequency = 0.
- Donot include background context or historical actions as events.

2) Event Extraction:
- You must output **only the most relevant event per industry**.
- Do NOT output duplicate event types across identical industry entries.
- If the same catalytic action applies to multiple industries, each receives its own version of that single event.

3) Evidence Requirement:
- Every event must include a verbatim “evidence” field directly quoting the article segment that proves the event.

4) Event_sentiment (Discrete: Positive, Negative, Neutral, Mixed)
- Positive: forward-looking benefit, growth, expansion, funding raised, approvals, major launches.
- Negative: layoffs, shutdowns, delays, investigations, fines, losses.
- Neutral: purely informational with no positive/negative impact language.
- Mixed: article contains both clear positive and negative impacts related to the same event.
- Always choose the sentiment most directly impacting the shortlisted industry.

5) Event_significance (Continuous score between 0.0 and 1.0)
- Represents the material importance of the event for the specific industry.
- Use the following scale:
    0.8-1.0 = Transformative, highly material, major corporate action, large funding, M&A, industry-wide impact.
    0.5-0.79 = Moderately material; strong impact but not transformative.
    0.2-0.49 = Mild or secondary impact; relevant but not major.
    0.0-0.19 = Minimal or weak impact; marginal relevance.
- Event significance MUST depend on the event's scale, magnitude, and proximity to the industry.

6) Market_reaction_likelihood (0-100)
- Probability that public markets or private investors would react to this event.
- Use this scale:
    80-100 = High likelihood of reaction (funding rounds, earnings, M&A, major deals, regulatory approvals).
    50-79 = Moderate likelihood (product launches, partnerships, operational changes).
    20-49 = Low likelihood (minor updates, local operational notes, soft commitments).
    0-19  = Very low likelihood (weak signals, context-only mentions).
- This MUST be an integer. 

7) Confidence (0.0-1.0)
- Your confidence in the correctness of the classifications for this event.
- Use the following rules:
    0.85-1.00 = Clear evidence, unambiguous event type, strong article support.
    0.60-0.84 = Some ambiguity or weak phrasing, but classification still well-supported.
    0.40-0.59 = Unclear or partial evidence; event type could be disputed.
    Below 0.40  = Avoid outputting the event altogether (too uncertain).
- Confidence MUST be based on clarity of evidence, not sentiment.

8) Event Frequency:
- event_frequency MUST be:
      1 if an event is identified for that industry,
      0 if no event is found.
- Do NOT count multiple mentions.
- Do NOT count background context or past actions.

9) No Duplicates:
- Merge all mentions describing the same catalytic action into the single selected event.
- If two shortlisted industries are identical (same sector + sub_category), treat them as one and output only one combined entry.

10) Emergent Industry Handling:
- If the user provides emergent_industry_candidate:
    - Check the article for clear, explicit evidence supporting that industry.
    - If evidence exists → keep the emergent industry candidate and give the most relevent event type for it based on retrieval from events.json.
 

"""

In [82]:
article_text1="""* TAG Heuer and New Balance launched a dual-release: the Connected Calibre E5 watch and FuelCell running shoe

* Both items share a violet/green color scheme and will be released in limited quantities worldwide

TAG Heuer and New Balance have joined forces to create a purpose-driven duo: the TAG Heuer Connected Calibre E5 40 mm x New Balance Edition smartwatch and the FuelCell SuperComp Elite v5 running shoe. Aimed at athletes who train with intention, this special-edition pairing underscores a shared obsession with precision, performance and pushing boundaries.

The TAG Heuer Connected Calibre E5 40 mm x New Balance Edition houses the brand's fifth-generation Calibre E5 movement and a proprietary in-house OS for an intuitive, responsive interface. Its black DLC-coated Grade-2 titanium case is sandblasted for lightweight durability, while a novel 0-100 scale on the dial syncs directly with app-based training metrics. "From the pinnacle design of our FuelCell SuperComp Elite 5 to the cutting-edge intelligence of TAG Heuer's Connected Calibre E5, we're creating instruments that elevate athletes and celebrate the art of innovation," notes Chris Davis, Global Brand President & CMO of New Balance.

Color and material choices further unite these performance icons. A deep violet hue - inspired by English Purple - and vibrant green accents mirror the co-branded New Balance shoe, while an engraved New Balance logo on the watch's sapphire caseback cements the precision partnership. TAG Heuer's patented Cushion Comfort System strap combines a flexible rubber base with high-tech textile from the shoe's inlay, delivering comfort and durability. Each watch, priced at 1,950 CHF (approx. $2,447 USD) also includes a second, New Balance-embroidered strap for a secure fit during intense workouts.

Built for race day, the $270 USD FuelCell SuperComp Elite v5 features an ultralight design with a carbon-fiber plate, PEBA midsole for explosive energy return and an 8 mm heel-to-toe drop. Engineered mesh uppers and a durable rubber outsole ensure breathability and traction when every second counts. "Our collaboration with New Balance felt like a natural alignment," adds George Ciz, TAG Heuer's CMO. "We've combined cutting-edge watch technology with the highest-performing running shoes to inspire people to push their limits -- on the track, the road or in life." Both items arrive in co-branded packaging with custom sockliners and will be released in limited quantities worldwide."""

In [81]:
article_text2="""NEW YORK (AP) -- Most U.S. stocks are ticking higher on Wednesday, a day after breaking their seven-day winning streak, while the price of gold is pushing further past $4,000 per ounce.

The S&P 500 rose 0.1% and is near its all-time high set on Monday. The Dow Jones Industrial Average was down 20 points, or less than 0.1%, as of 9:35 a.m. Eastern time, while the Nasdaq composite was 0.3% higher.

AST SpaceMobile jumped 17% after Verizon Communications agreed to use its space-based network to offer service to its cellular customers when needed, starting in 2026. Verizon edged down by 0.1%.

The stock market is near records, but trading has been relatively muted recently following the U.S. government's latest shutdown. That has delayed the release of several major economic reports, such as the monthly jobs report, which usually move the market. Stocks have been drifting without them or other signals to change forecasts for continued cuts to interest rates by the Federal Reserve, one of the major reasons for the stock market's stellar performance since April.

Another force that's pushed the market to records is the frenzy around artificial-intelligence technology.

Poet Technologies rallied 4.7% and added to its surge of 23.5% from Tuesday, when it said it raised $75 million in investment to accelerate its growth. The company sells high-speed optical engines and other products used in the AI systems market.

Dell Technologies likewise piled more gains onto its own rally from Tuesday, when it talked up growth opportunities related to AI, and rose nearly 6% for the biggest gain in the S&P 500.

AI-related stocks have broadly been on a tear. Chipmaker Nvidia came into the day up nearly 40% for the year so far. Oracle soared 70% over the same time, while Palantir Technologies more than doubled with a 140.9% surge.

But such performances have raised criticism that prices have gone too far, too fast and are reminiscent of the 2000 dot-com bubble that ultimately imploded and halved the S&P 500's value.

Proponents say AI stocks are backed by big growth in profits, something that many dot-com stocks didn't have. But the Bank of England nevertheless warned Wednesday of the rising risk that tech stock share prices pumped up by the AI boom could face a "sudden correction."

"On a number of measures, equity market valuations appear stretched, particularly for technology companies focused on Artificial Intelligence," policymakers at the U.K. central bank said in a report. With Big Tech companies accounting for an increasingly outsize share of stock market indexes, stocks are "particularly exposed should expectations around the impact of AI become less optimistic."

Gold, meanwhile, continued its stellar year and rose further past $4,000 per ounce. Investors have traditionally seen gold as a way to protect against rising inflation, and its price has soared more than 50% this year.

Worries are high about big debt loads that the U.S. and other governments are building, which threaten to push inflation higher. Political instability around the world and expectations for rate cuts by the Fed are also helping to push up interest in gold.

The Fed will release the minutes from its last meeting later in the afternoon. That's when the Fed cut its main interest rate for the first time this year and hinted that more may be on the way. It's trying to support a slowing job market but also says it's staying mindful of inflation, which remains above its target of 2%. Lower rates can give inflation more fuel.

In stock markets abroad, indexes rose in Europe following a weaker finish in Asia.

In the bond market, the yield on the 10-year Treasury eased to 4.10% from 4.14% late Tuesday.

___

AP Business Writers Matt Ott, Elaine Kurtenbach and Kelvin Chan contributed."""

In [80]:
article_text3="""The buyout is part of Sun Holdings' strategy to expand its nationwide presence in casual dining and to diversify its portfolio. Sun Holdings started out primarily as a franchisee but has since built up a significant portfolio through unit development, acquisitions and brand ownership, the press release said.

Sun Holdings began its acquisition streak in 2019 through its purchase of distressed Taco Bueno, which also underwent bankruptcy. It then bought Freebirds World Burrito in 2024, and acquired Uncle Julio's during a foreclosure auction earlier this year, creating a portfolio of Mexican food brands. The holding company now operates over 1,800 locations nationwide, including Applebee's, Arby's, Burger King, Golden Corral, IHOP, McAlister's, Papa John's and Popeyes locations. Bar Louie is Sun Holdings' first gastropub brand.

"This acquisition marks an exciting step forward in our growth strategy," said Guillermo Perales, president and CEO of Sun Holdings. "Bar Louie is renowned for its signature martinis and innovative, craveable food. Its unique brand identity makes it an ideal addition to our casual dining portfolio and highlights our strategy of pairing operational expertise with growth opportunities."

But the Bar Louie brand is likely to pose major challenges for Sun Holdings. The chain tried to jump start franchising in 2023, when it had 67 units, by offering discounted franchising fees for first-time operators. But by the end of that year, it projected only three franchised restaurant openings in 2024, according to its franchise disclosure document. Coming out of bankruptcy, the chain's store count dropped to 39 units -- slightly more than half of what it had in 2021."""

In [79]:
article_text4="""SÃO PAULO, Oct. 8, 2025 /PRNewswire/ -- AeroCore Technologies announced today the opening of its first international site at São Paulo/Guarulhos International Airport (GRU). AeroCore is a global aviation technology company specializing in advanced engine cleaning and performance optimization. This expansion marks the beginning of AeroCore's worldwide rollout of its Total Engine Health™ solutions, designed to help airlines improve fuel efficiency, reduce emissions, and extend engine life.

The GRU site will serve as AeroCore's South American hub, offering its patented Nucleated Foam Technology™, a breakthrough solution that delivers measurable improvements in exhaust gas temperature (EGT) margins, fuel efficiency, and maintenance cycle times compared to conventional engine wash methods.

"GRU is one of the fastest-growing airports in Latin America and a critical hub for both domestic and international aviation," said Kevin Bourke, CEO and Co-Founder of AeroCore Technologies."Establishing operations here reflects our commitment to supporting airline partners across the Americas with proven technology that enhances performance, reduces environmental impact, and drives long-term value."

AeroCore's international expansion builds on its success with commercial airlines and defense operators. Looking ahead, the company is currently opening additional sites in Hawaii, Japan, and the Middle East, extending its global footprint and reinforcing its mission to deliver Total Engine Health™ wherever aircraft operate.

About AeroCore Technologies

AeroCore Technologies delivers advanced engine cleaning and maintenance solutions through its proprietary Nucleated Foam Technology™. With headquarters in the United States and operations expanding globally, AeroCore helps airlines and defense operators improve fuel efficiency, restore engine performance, and reduce carbon emissions. More than 50% of AeroCore's workforce is made up of military veterans, underscoring the company's commitment to excellence, service, and efficiency.

In addition to aviation, AeroCore's Nucleated Foam Technology™ delivers significant performance improvements in the O&G sector through its joint venture, ReNu Technologies.

For more information about AeroCore Technologies, please visit www.aerocore.com or contact [email protected]; for ReNu Technologies, please visit www.renutech.com or contact [email protected]"""

In [50]:
article_text5="""Looking at the universe of stocks we cover at Dividend Channel, on 10/10/25, Lennar Corp (Symbol: LEN), American Eagle Outfitters, Inc. (Symbol: AEO), and Darden Restaurants, Inc. (Symbol: DRI) will all trade ex-dividend for their respective upcoming dividends. Lennar Corp will pay its quarterly dividend of $0.50 on 10/27/25, American Eagle Outfitters, Inc. will pay its quarterly dividend of $0.125 on 10/29/25, and Darden Restaurants, Inc. will pay its quarterly dividend of $1.50 on 11/3/25. As a percentage of LEN's recent stock price of $122.14, this dividend works out to approximately 0.41%, so look for shares of Lennar Corp to trade 0.41% lower -- all else being equal -- when LEN shares open for trading on 10/10/25. Similarly, investors should look for AEO to open 0.81% lower in price and for DRI to open 0.78% lower, all else being equal.

Below are dividend history charts for LEN, AEO, and DRI, showing historical dividends prior to the most recent ones declared.

In general, dividends are not always predictable, following the ups and downs of company profits over time. Therefore, a good first due diligence step in forming an expectation of annual yield going forward, is looking at the history above, for a sense of stability over time. This can help in judging whether the most recent dividends from these companies are likely to continue. If they do continue, the current estimated yields on annualized basis would be 1.64% for Lennar Corp, 3.23% for American Eagle Outfitters, Inc., and 3.13% for Darden Restaurants, Inc..

In Wednesday trading, Lennar Corp shares are currently down about 3.4%, American Eagle Outfitters, Inc. shares are off about 1.8%, and Darden Restaurants, Inc. shares are down about 0.9% on the day.

Click here to learn which 25 S.A.F.E. dividend stocks should be on your radar screen "
"""

In [59]:
article_text6="""Toll Brothers (TOL) is one of the stocks most watched by Zacks.com visitors lately. So, it might be a good idea to review some of the factors that might affect the near-term performance of the stock.

Over the past month, shares of this home builder have returned +4.3%, compared to the Zacks S&P 500 composite's -4.5% change. During this period, the Zacks Building Products - Home Builders industry, which Toll Brothers falls in, has gained 18.4%. The key question now is: What could be the stock's future direction?

While media releases or rumors about a substantial change in a company's business prospects usually make its stock 'trending' and lead to an immediate price change, there are always some fundamental facts that eventually dominate the buy-and-hold decision-making.

Rather than focusing on anything else, we at Zacks prioritize evaluating the change in a company's earnings projection. This is because we believe the fair value for its stock is determined by the present value of its future stream of earnings.

Our analysis is essentially based on how sell-side analysts covering the stock are revising their earnings estimates to take the latest business trends into account. When earnings estimates for a company go up, the fair value for its stock goes up as well. And when a stock's fair value is higher than its current market price, investors tend to buy the stock, resulting in its price moving upward. Because of this, empirical studies indicate a strong correlation between trends in earnings estimate revisions and short-term stock price movements.

Toll Brothers is expected to post earnings of $3.28 per share for the current quarter, representing a year-over-year change of -12.1%. Over the last 30 days, the Zacks Consensus Estimate remained unchanged.

The consensus earnings estimate of $14.11 for the current fiscal year indicates a year-over-year change of +14.2%. This estimate has remained unchanged over the last 30 days.

For the next fiscal year, the consensus earnings estimate of $13.83 indicates a change of -2% from what Toll Brothers is expected to report a year ago. Over the past month, the estimate has remained unchanged.

With an impressive externally audited track record, our proprietary stock rating tool -- the Zacks Rank -- is a more conclusive indicator of a stock's near-term price performance, as it effectively harnesses the power of earnings estimate revisions. The size of the recent change in the consensus estimate, along with three other factors related to earnings estimates, has resulted in a Zacks Rank #3 (Hold) for Toll Brothers."""

In [75]:
industry_object1={
    "shortlisted_industries": [
        {
            "sector": "Sports, Fitness & Wellness",
            "sub_category": "Sports Tech",
            "relevance": "HIGH",
            "reason": "Central focus on a performance-oriented smartwatch launch for athletes as part of a sports collaboration."
        },
        {
            "sector": "Sports, Fitness & Wellness",
            "sub_category": "Connected Fitness",
            "relevance": "MEDIUM",
            "reason": "Smartwatch features proprietary OS and app-based training metrics, highlighting connected training use cases."
        },
        {
            "sector": "Materials & Mining",
            "sub_category": "Specialty Materials",
            "relevance": "LOW",
            "reason": "Product details emphasize advanced materials like DLC-coated titanium, carbon-fiber plate, PEBA midsole, and sapphire caseback."
        }
    ],
    "emergent_industry_candidate": None
}

In [76]:
industry_object2={
    "shortlisted_industries": [
        {
            "sector": "Technology & Software",
            "sub_category": "Semiconductors (Hardware)",
            "relevance": "MEDIUM",
            "reason": "Article emphasizes AI-driven market moves and highlights chipmaker Nvidia’s gains, making semiconductor hardware a substantive thread impacting markets. Retrieved from vector store."
        }
    ],
    "emergent_industry_candidate": None
}

In [77]:
industry_object3={
  "shortlisted_industries": [
    {
      "sector": "Consumer Goods & Retail",
      "sub_category": "Food & Beverage",
      "relevance": "HIGH",
      "reason": "Article centres on Sun Holdings' acquisition of Bar Louie and expansion of its casual dining/restaurant portfolio, franchising and operations. Retrieved from vector store."
    }
  ],
  "emergent_industry_candidate": None
}

In [78]:
industry_object4={
  "shortlisted_industries": [
    {
      "sector": "Aerospace & Defense",
      "sub_category": "MRO (Maintenance, Repair & Overhaul)",
      "relevance": "HIGH",
      "reason": "Article describes AeroCore's engine cleaning and Total Engine Health solutions for airlines and defense operators and the opening of an international service site at GRU — core MRO activity. Retrieved from vector store."
    }
  ],
  "emergent_industry_candidate": None
}

In [54]:
industry_object_5={
  "shortlisted_industries": [
    {
      "sector": "Consumer Goods & Retail",
      "sub_category": "E-commerce",
      "relevance": "LOW",
      "reason": "American Eagle Outfitters, a retailer explicitly mentioned as paying a dividend, links the article to the consumer retail sector. Retrieved from vector store."
    }
  ],
  "emergent_industry_candidate": None
}

In [61]:
industry_object6={
    "shortlisted_industries": [
        {
            "sector": "Real Estate & Construction",
            "sub_category": "Commercial Real Estate",
            "relevance": "HIGH",
            "reason": "The article focuses on Toll Brothers, a home builder and its performance within the building-products/home builders industry, making Real Estate & Construction central to the piece. Retrieved from vector store."
        }
    ],
    "emergent_industry_candidate": None
}

In [71]:
USER_PROMPT=f"""Analyze the article & the industry object below to extract **only the  most relevant catalytic events for EACH shortlisted industry**, following the System Prompt rules.

Article:
{article_text7}

Industry object:
{industry_object7}

Requirements:
- Extract exactly the most relevant event per industry (or zero if no relevant event exists).
- Only score event explicitly tied to the shortlisted industry/sub-category.
- Use explicit evidence from the article & file search results to give event extraction.
"""

In [63]:
textt_format = {
    "format": {
        "type": "json_schema",
        "name": "IndustryEventExtraction",
        "schema": {
            "type": "object",
            "properties": {
                "shortlisted_industries": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "sector": { "type": "string" },
                            "sub_category": { "type": "string" },
                            "events": {
                                "type": "array",
                                "items": {
                                    "type": "object",
                                    "properties": {
                                        "event_type": {
                                            "type": "string",
                                        },
                                        "event_sentiment": {
                                            "type": "string",
                                            "enum": ["Positive", "Negative", "Mixed", "Neutral"]
                                        },
                                        "event_significance": {
                                            "type": "number",
                                            "minimum": 0,
                                            "maximum": 10
                                        },
                                        "market_reaction_likelihood": {
                                            "type": "integer",
                                            "minimum": 0,
                                            "maximum": 100
                                        },
                                        "confidence": {
                                            "type": "number",
                                            "minimum": 0,
                                            "maximum": 100
                                        },
                                        "evidence": {
                                            "type": "string"
                                        }


                                    },
                                    "required": [
                                        "event_type",
                                        "event_sentiment",
                                        "event_significance",
                                        "market_reaction_likelihood",
                                        "confidence",
                                        "evidence"
                                    ],
                                    "additionalProperties": False
                                }
                            },
                            "event_frequency": {
                                "type": "integer",
                                "minimum": 0
                            }
                        },
                        "required": ["sector", "sub_category", "events", "event_frequency"],
                        "additionalProperties": False
                    }
                },
                "emergent_industry_candidate": {
                    "anyOf": [
                        {
                            "type": "object",
                            "properties": {
                                "label": { "type": "string" },
                                "rationale": { "type": "string" },
                                "event_type":{
                                            "type": "string",
                                            
                                        },
                                        "event_sentiment": {
                                            "type": "string",
                                            "enum": ["Positive", "Negative", "Mixed", "Neutral"]
                                        },
                                        "event_significance": {
                                            "type": "number",
                                            "minimum": 0,
                                            "maximum": 10
                                        },
                                        "market_reaction_likelihood": {
                                            "type": "integer",
                                            "minimum": 0,
                                            "maximum": 100
                                        },
                                        "confidence": {
                                            "type": "number",
                                            "minimum": 0,
                                            "maximum": 100
                                        },
                                        "evidence": {
                                            "type": "string"
                                        }
                            },
                            "required": ["label", "rationale","event_type", "event_sentiment", "event_significance", "market_reaction_likelihood", "confidence", "evidence"],
                            "additionalProperties": False
                        },
                        { "type": "null" }
                    ]
                }
            },
            "required": ["shortlisted_industries", "emergent_industry_candidate"],
            "additionalProperties": False
        },
        "strict": True
    }
}


In [72]:
response = client.responses.create(
        model="gpt-5-mini",
        
         tools=[
                {"type": "file_search",
                 "vector_store_ids": ["vs_693087a5f7b88191aea544ef5ab0560b"],
                }
        ],
        text=textt_format,
        reasoning={"effort": "high"},
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT}
        ],                    
)

# Get text output
output_text = getattr(response, "output_text", None)
if not output_text:
        blocks = []
        for item in getattr(response, "output", []) or []:
            for c in getattr(item, "content", []) or []:
                if isinstance(c, dict) and c.get("type") == "output_text":
                    blocks.append(c.get("text", ""))
        output_text = "".join(blocks) if blocks else ""

if not output_text:
      raise RuntimeError("No text output returned by the model.")

try:
    print(json.loads(output_text))
except json.JSONDecodeError as e:
        raise RuntimeError(f"Model returned non-JSON output: {output_text}") from e


{'shortlisted_industries': [{'sector': 'Entertainment & Media', 'sub_category': 'Streaming Services', 'events': [{'event_type': 'Product Launch', 'event_sentiment': 'Positive', 'event_significance': 0.35, 'market_reaction_likelihood': 30, 'confidence': 0.95, 'evidence': 'To preview the set, Dylan\'s label has released an outtake of "Rocks and Gravel" recorded in studio on April 25, 1962 to streaming services and YouTube. '}], 'event_frequency': 1}], 'emergent_industry_candidate': None}


In [73]:
try:
    parsed = json.loads(output_text)
   
    pretty_json = json.dumps(parsed, indent=4, ensure_ascii=False)

    print(pretty_json)

except json.JSONDecodeError as e:
    print("Invalid JSON:", e)

{
    "shortlisted_industries": [
        {
            "sector": "Entertainment & Media",
            "sub_category": "Streaming Services",
            "events": [
                {
                    "event_type": "Product Launch",
                    "event_sentiment": "Positive",
                    "event_significance": 0.35,
                    "market_reaction_likelihood": 30,
                    "confidence": 0.95,
                    "evidence": "To preview the set, Dylan's label has released an outtake of \"Rocks and Gravel\" recorded in studio on April 25, 1962 to streaming services and YouTube. "
                }
            ],
            "event_frequency": 1
        }
    ],
    "emergent_industry_candidate": null
}


In [ ]:
# TOKEN USAGE
print("\n--- TOKEN USAGE ---")
print("Input tokens: ", response.usage.input_tokens)
print("Output tokens:", response.usage.output_tokens)


--- TOKEN USAGE ---
Input tokens:  5244
Output tokens: 4537
